In [1]:
!pip install -q python-dotenv pandas tqdm openai


In [2]:
import os
import json
import re
import time
import random
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI


In [3]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"


In [4]:
CSV_PATH = "../datasets/TWeddit.csv"

df = pd.read_csv(CSV_PATH)

# Drop stray index columns like "Unnamed: 0"
unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if unnamed_cols:
    df.drop(columns=unnamed_cols, inplace=True)

print(df.shape)
df.head()


(4994, 5)


,id,subreddit,title,body,Tags
0,1ljxynj,abortion,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...","Medical, Pregnancy, Mental Health"
1,1ljxtt8,abortion,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,"Medical, Pregnancy, Mental Health"
2,1ljwhkb,abortion,Help needed/ live in Texas where abortion in b...,Anyone know of a legit site to support women i...,Medical
3,1ljvy6u,abortion,medical abortion at 6 weeks,I’ll be doing my procedure on Friday and I got...,Medical
4,1ljv5k4,abortion,Idk what to feel about my decision after doing...,I just had medical abortion yesterday. I was a...,"Pregnancy, Mental Health"


In [6]:
required_cols = ["id", "subreddit", "title", "body", "Tags"]
missing = [c for c in required_cols if c not in df.columns]
print("Required:", required_cols)
print("Missing:", missing)


Required: ['id', 'subreddit', 'title', 'body', 'Tags']
Missing: []


In [7]:
if "Gender" not in df.columns:
    df["Gender"] = None

if "Gender_meta" not in df.columns:
    df["Gender_meta"] = None

df["Gender"].value_counts(dropna=False)


Gender
None    4994
Name: count, dtype: int64

In [15]:
def build_prompt(title, body):
    return f"""
You are an expert in inferring gender from text. Classify the gender of the PRIMARY PERSON being described in the post.

The primary person is:
- The person whose experience/story the post mainly discusses (often the narrator "I/me", but it could be a friend/partner/sister, etc.).
- If the post is clearly about someone else (e.g., "my friend", "my boyfriend", "my sister"), infer the gender of that person.
- If there are multiple people and it's not clear who the primary subject is, choose "Can't Infer".

Valid labels:
- "Male"
- "Female"
- "Gender Fluid"  (non-binary, trans, intersex, exploring gender, multiple gender identities)
- "Can't Infer"

Rules:
- Prefer explicit self-identification when present (e.g., "I am a woman", "as a guy", "I'm nonbinary").
- You MAY infer "Female" if there are strong reproductive/anatomical cues like:
  uterus/ovaries/cervix, menstrual periods, pregnancy, gynecologist visits, etc.
- You MAY infer "Male" if there are strong cues like:
  prostate/testicles, explicit statements of being male, etc.
- If meaningful ambiguity remains → "Can't Infer".

Return ONLY JSON:
{{"gender": "<label>"}}

Post:
Title: {title}
Body: {body}
"""


In [16]:
def truncate_text(s, max_chars=3000):
    s = "" if s is None else str(s)
    return s[:max_chars]

def extract_json_object(text: str):
    if not text:
        return None
    m = re.search(r"\{[\s\S]*\}", text)
    return m.group(0) if m else None


In [17]:
def classify_gender_with_meta(title, body, max_retries=4):
    body = truncate_text(body, 3000)
    prompt = build_prompt(title, body)

    last_err = None
    last_raw = ""

    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=120,
            )

            raw = (resp.choices[0].message.content or "").strip()
            last_raw = raw

            if not raw:
                raise ValueError("Empty model output")

            json_text = extract_json_object(raw)
            if not json_text:
                raise ValueError("No JSON object found")

            data = json.loads(json_text)
            gender = data.get("gender", "Can't Infer")

            allowed = ["Male", "Female", "Gender Fluid", "Can't Infer"]
            gender = gender if gender in allowed else "Can't Infer"

            return gender, {
                "ok": True,
                "attempts": attempt,
                "error": None,
                "raw": raw[:300],
            }

        except Exception as e:
            last_err = str(e)
            # Exponential backoff + jitter
            sleep_s = min(12, (2 ** (attempt - 1)) + random.random())
            time.sleep(sleep_s)

    return "Can't Infer", {
        "ok": False,
        "attempts": max_retries,
        "error": last_err,
        "raw": last_raw[:300],
    }


In [18]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    temperature=0,
    max_tokens=10,
)
print("RAW:", repr(resp.choices[0].message.content))


RAW: 'OK'


In [19]:
sample_title = "Just found out who it was that blackmailed my friend with pornographic content he coerced her to make"
sample_body = """
    [removed]
"""

gender, meta = classify_gender_with_meta(sample_title, sample_body)
print("Predicted Gender:", gender)
print("Meta:", meta)


Predicted Gender: Female
Meta: {'ok': True, 'attempts': 1, 'error': None, 'raw': '{"gender": "Female"}'}


In [20]:
SAVE_PATH = "../datasets/TWeddit_with_gender.csv"

if os.path.exists(SAVE_PATH):
    df = pd.read_csv(SAVE_PATH)
    print("✅ Resumed from:", SAVE_PATH)
else:
    print("🆕 Starting fresh. Will save to:", SAVE_PATH)


🆕 Starting fresh. Will save to: ../datasets/TWeddit_with_gender.csv


In [21]:
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if pd.notna(row.get("Gender")) and str(row["Gender"]).strip() != "":
        continue

    title = str(row.get("title", ""))
    body = str(row.get("body", ""))

    gender, meta = classify_gender_with_meta(title, body)
    df.at[idx, "Gender"] = gender
    df.at[idx, "Gender_meta"] = json.dumps(meta)

    # Save every 50 rows
    if idx % 50 == 0:
        df.to_csv(SAVE_PATH, index=False)

df.to_csv(SAVE_PATH, index=False)
print("✅ Done. Saved to:", SAVE_PATH)


100%|██████████| 4994/4994 [44:10<00:00,  1.88it/s]  


✅ Done. Saved to: ../datasets/TWeddit_with_gender.csv
